In [ ]:
# ===== 第 1 块：读取数据 + 画原始峰 =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# ---- 如果路径或列名不一样，只改这三行 ----
file_path  = r'C:\Users\Administrator\Desktop\GPC_analysis\WJ38688\Peak2.csv'
time_col   = 'Time'      # 时间那一列的列名
signal_col = 'Signal'    # 信号那一列的列名
# ------------------------------------------

data = pd.read_csv(file_path)
print(data.head())       # 打印前几行，确认列名对不对

time   = data[time_col].values
signal = data[signal_col].values

plt.figure(figsize=(10, 4))
plt.plot(time, signal, 'b-', label='GPC峰')
plt.xlabel('保留时间 (min)')
plt.ylabel('信号强度')
plt.title('GPC原始峰')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# ===== 第 2 块：计算 Mw、Mn、PDI =====

# ---- 标定曲线系数：log10(M) = a + b*t + c*t² + d*t³ ----
a = 20.2594
b = -1.5829
c = 0.05776
d = -0.000794689

# ---- 从上图读出的峰起止时间，改成你自己的值 ----
t_start = 28.0    # 峰开始的时间
t_end   = 32.0    # 峰结束的时间
dt      = 0.05    # 切片间隔，越小越精细（0.05 或 0.1 都行）
# ------------------------------------------------

# 根据保留时间算分子量 M
def calc_M(t):
    logM = a + b*t + c*t**2 + d*t**3
    return 10**logM

# 在峰范围内切片，并算出每个切片点的 M 和信号强度
t_slices      = np.arange(t_start, t_end, dt)
M_slices      = calc_M(t_slices)
signal_slices = np.interp(t_slices, time, signal)

# ---- 核心计算（GPC 信号 ∝ 重量 w）----
Mw  = np.sum(signal_slices * M_slices) / np.sum(signal_slices)        # 重均分子量
Mn  = np.sum(signal_slices) / np.sum(signal_slices / M_slices)        # 数均分子量
PDI = Mw / Mn                                                         # 多分散系数

print(f"Mw  = {Mw:,.0f}")
print(f"Mn  = {Mn:,.0f}")
print(f"PDI = {PDI:.3f}")

# ---- 验证图：看看切片点有没有盖住整个峰 ----
plt.figure(figsize=(8, 4))
plt.plot(time, signal, 'b-', label='原始峰')
plt.scatter(t_slices, signal_slices, color='red', s=10, label='切片点')
plt.axvline(t_start, color='gray', ls='--')
plt.axvline(t_end,   color='gray', ls='--')
plt.xlabel('保留时间 (min)')
plt.ylabel('信号强度')
plt.title('峰与切片范围')
plt.legend()
plt.show()